# पाठ 10 - उत्पादनमा AI एजेन्टहरू

यस पाठमा तपाईं Microsoft Agent Framework (Python) प्रयोग गरेर AI एजेन्टहरूको लागि **उत्पादन ढाँचाहरू** सिक्नुहुनेछ। हामीले समेटेका विषयहरू:

- **अवलोकनयोग्यता** — एजेन्ट अन्तरक्रियाहरूमा समय मापन र लगिङ थप्ने
- **मूल्याङ्कन** — उत्तरको गुणस्तर स्कोर गर्न मूल्याङ्कन एजेन्ट प्रयोग गर्ने
- **लागत व्यवस्थापन** — टोकन अनुकूलन र मोडल छनोटका रणनीतिहरू

परिदृश्य एउटा **यात्रा एजेन्ट** हो जसले प्रयोगकर्ताहरूलाई यात्राको योजना बनाउन मद्दत गर्छ, र माथि निगरानी तथा मूल्याङ्कन परत थपिएको छ।


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## उत्पादन सम्बन्धी विचारहरू

AI एजेन्टहरूलाई प्रोटोटाइपबाट उत्पादनमा सार्दा तीन स्तम्भहरूमा सावधानीपूर्वक ध्यान दिनुपर्छ:

1. **अवलोकनयोग्यता** — तपाईंले एजेन्टले के गर्दैछ, कति समय लाग्छ, र कुन उपकरणहरू प्रयोग गर्दछ भन्ने कुरामा दृश्यता चाहिन्छ। ट्रेसिङ र लगिङ बिना, उत्पादन समस्याहरू डिबग गर्नु लगभग असम्भव हुन्छ।

2. **मूल्यांकन** — स्वचालित गुणस्तर जाँचहरूले सुनिश्चित गर्छन् कि एजेन्टका प्रतिक्रियाहरू समयसँगै सही, पूर्ण, र उपयोगी रहन्छन्। एउटा मूल्यांकन गर्ने एजेन्टले परिभाषित मापदण्डहरू विरुद्ध प्रतिक्रियाहरूलाई स्कोर गर्न सक्छ।

3. **लागत व्यवस्थापन** — टोकन प्रयोगले प्रत्यक्ष रूपमा लागतमा प्रभाव पार्छ। प्रॉम्प्ट अनुकूलन, मोडेल छनोट, र क्यासिङ जस्ता रणनीतिहरूले गुणस्तर सम्झौता नगरी खर्चलाई नियन्त्रणमा राख्न मद्दत गर्छन्।


## अवलोकनयोग्य एजेन्ट बनाउने

हामी यात्रा उपकरणहरू परिभाषित गर्छौं र एजेन्ट कलमा समय मापन थपेर यसलाई घेर्छौं ताकि हामी ढिलाइ निगरानी गर्न सकौं। उत्पादनमा तपाईंले OpenTelemetry वा समान ट्रेसिङ ब्याकएन्डसँग एकीकृत गर्नुहुनेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन ढाँचाहरू

सामान्य उत्पादन ढाँचाको एउटा प्रचलित तरिका भनेको दोस्रो एजेन्टलाई **मूल्यांकनकर्ता** को रूपमा प्रयोग गर्नु हो। मूल्यांकनकर्ताले प्राथमिक एजेन्टको प्रतिक्रियालाई पूर्वनिर्धारित मापदण्डहरू जस्ता पूर्णता, शुद्धता, र सहायकताको आधारमा अंक दिन्छ।

यसले सक्षम बनाउँछ:
- प्रयोगकर्तासम्म उत्तर पुग्नु अघि स्वचालित गुणस्तर गेटहरू
- प्रम्प्टहरू वा मोडेलहरू परिवर्तन हुँदा रिग्रेसन पत्ता लगाउने
- समयसँगै एजेन्ट प्रदर्शनको निरन्तर अनुगमन


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## लागत व्यवस्थापन रणनीतिहरू

उत्पादन AI एजेन्टहरूको लागि खर्च नियन्त्रण महत्वपूर्ण छ। यहाँ मुख्य रणनीतिहरू छन्:

| रणनीति | विवरण |
|---|---|
| **प्रॉम्प्ट अनुकूलन** | प्रणाली निर्देशहरू संक्षिप्त राख्नुहोस्। इनपुट टोकन घटाउन फालतु सन्दर्भहरू हटाउनुहोस्। |
| **मोडेल चयन** | वर्गीकरण वा निष्कर्षण जस्ता साधारण कार्यहरूका लागि साना, सस्ता मोडेलहरू (जस्तै GPT-4o-mini) प्रयोग गर्नुहोस्, र जटिल तर्कका लागि ठूलो मोडेलहरू राख्नुहोस्। |
| **क्याचिङ** | दोहोरिने API कलहरूबाट बच्नका लागि उपकरण परिणामहरू र बारम्बार गरिएको क्वेरीहरू क्याच गर्नुहोस्। |
| **टोकन बजेट** | अप्रत्याशित रूपमा लामो प्रतिक्रियाहरू रोक्न `max_tokens` सीमाहरू सेट गर्नुहोस्। |
| **ब्याचिङ** | सम्भव भएमा धेरै प्रयोगकर्ता क्वेरीहरूलाई एकल API कलमा समूहबद्ध गर्नुहोस्। |

व्यवहारमा, स्तरबद्ध दृष्टिकोण राम्रो काम गर्छ: सरल अनुरोधहरूलाई छिटो र सस्तो मोडेलमा मार्गनिर्देश गर्नुहोस् र केवल जटिल क्वेरीहरूलाई मात्र अधिक क्षमताशाली मोडेलमा उकास्नुहोस्।


## सारांश

यस पाठमा तपाईंले निम्न कुराहरू सिक्नुभयो:

1. **अवलोकनीयता थप्नुहोस्** एजेन्ट अन्तरक्रियाहरूमा समयमा ट्र्याकिङ र लगिङ समावेश गरेर, ट्रेसिङ र मोनिटरिङका लागि आधार तयार पार्दै।
2. **एजेन्ट प्रतिक्रियाहरू मूल्याङ्कन गर्नुहोस्** स्वतः मूल्याङ्कन गर्ने एजेन्ट प्रयोग गरी जसले पूर्णता, सटीकता, र सहायकतालाई स्कोर गर्छ।
3. **लागत व्यवस्थापन गर्नुहोस्** प्रम्प्ट अनुकूलन, मोडेल छनौट, क्यासिङ, र टोकन बजेटहरू मार्फत।

यी उत्पादन ढाँचाहरूले तपाईंका एआई एजेन्टहरूलाई स्केलमा भरपर्दो, मापनयोग्य, र लागत-प्रभावी बनाउने सुनिश्चित गर्न मद्दत गर्छन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यो कागजात AI अनुवाद सेवा Co-op Translator (https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी सटीकताको लागि प्रयासरत भए तापनि, कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटि वा अशुद्धता हुन सक्छ। मूल भाषामा रहेको दस्तावेजलाई प्रामाणिक स्रोत मानिनु पर्छ। महत्त्वपूर्ण जानकारीको लागि व्यावसायिक मानवीय अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट भएका कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
